# GSE544 Homework 5 Part 2
**Bishwa R. Rai with Claude Sonnet 4.6**  
2026-05-11

*LLM disclaimer: Claude Sonnet 4.6 was used to assist in writing code and to check solutions.*  
*The lecture notebook `4_copula_nco.ipynb` was used as a starting point.*

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import pickle, os

RNG_SEED = 778
N_MC     = 500_000
EPS      = 1e-9
K        = 5   # clusters

## Load model and data

In [ ]:
with open("copula_model.pkl", "rb") as fh:
    M = pickle.load(fh)

marginals_all = M["marginals"]
chol_all      = M["chol_P"]
cap_all       = M["cap"]
p             = M["n_assets"]

returns = pd.read_csv("vmls_portfolio_returns.csv").to_numpy()
R       = returns[:2000, :19]
R_test  = returns[2000:, :19]
rf_bar  = returns[:2000, 19].mean()
T_test  = R_test.shape[0]
print(f"R {R.shape}  R_test {R_test.shape}  rf_bar {rf_bar:.3e}/day")

## Constrained CRRA optimizer

Three changes from the lecture's `kelly_opt`:
1. **No shorting**: bounds `w_j >= 0`
2. **No borrowing**: constraint `sum(w) <= 1`
3. **CRRA utility**: `u_gamma(x) = x^(1-gamma)/(1-gamma)` for `gamma != 1`; log utility for `gamma = 1`

Gross growth is still `g_t(w) = b(w)*rf_bar + R @ w` with `b(w) = 1 - sum(w) >= 0`.

In [ ]:
def gross_growth(w, Rm):
    w = np.asarray(w, float)
    return (1.0 - w.sum()) * rf_bar + Rm @ w

def crra_opt(Rm, gamma):
    """Constrained CRRA optimizer: long-only, sum(w) <= 1."""
    n = Rm.shape[1]
    X = Rm - rf_bar
    base = 1.0 + rf_bar

    if gamma == 1:
        # log utility, same as lecture kelly_opt but with constraints
        def f(w):
            a = np.maximum(base + X @ w, EPS)
            return -np.mean(np.log(a))
        def grad(w):
            a = np.maximum(base + X @ w, EPS)
            return -(X / a[:, None]).mean(axis=0)
    else:
        def f(w):
            a = np.maximum(base + X @ w, EPS)
            return -np.mean(a ** (1 - gamma) / (1 - gamma))
        def grad(w):
            a = np.maximum(base + X @ w, EPS)
            return -(X * a[:, None] ** (-gamma)).mean(axis=0)

    bounds     = [(0.0, None)] * n
    constraint = {"type": "ineq",
                  "fun":  lambda w: 1.0 - w.sum(),
                  "jac":  lambda w: -np.ones(n)}

    res = minimize(f, np.full(n, 0.5 / n), jac=grad,
                   method="SLSQP",
                   bounds=bounds,
                   constraints=constraint,
                   options=dict(maxiter=4000, ftol=1e-12))
    return res.x

def mean_log_growth(w, Rm):
    a = np.maximum(1.0 + gross_growth(w, Rm), EPS)
    return np.mean(np.log(a))

def realized_crra(w, Rm, gamma):
    a = np.maximum(1.0 + gross_growth(w, Rm), EPS)
    if gamma == 1:
        return np.mean(np.log(a))
    return np.mean(a ** (1 - gamma) / (1 - gamma))

## Copula simulation helpers (unchanged from lecture)

In [ ]:
def _fit_marginals_and_corr(Mtx):
    k   = Mtx.shape[1]
    mar = [stats.t.fit(Mtx[:, j]) for j in range(k)]
    U   = np.clip(np.column_stack([stats.t.cdf(Mtx[:, j], *mar[j])
                                   for j in range(k)]), 1e-6, 1 - 1e-6)
    Z   = stats.norm.ppf(U)
    Pc  = np.atleast_2d(np.corrcoef(Z, rowvar=False)) if k > 1 else np.array([[1.0]])
    ev, V = np.linalg.eigh(Pc)
    Pc  = V @ np.diag(np.clip(ev, 1e-10, None)) @ V.T
    d   = np.sqrt(np.diag(Pc))
    Pc  = Pc / np.outer(d, d)
    return mar, np.linalg.cholesky(Pc)

def fit_gaussian_copula(Mtx):
    mar, L = _fit_marginals_and_corr(Mtx)
    cap = 2.0 * np.max(np.abs(Mtx), axis=0)
    return dict(mar=mar, L=L, cap=cap, k=Mtx.shape[1])

def simulate(model, n, seed):
    k  = model["k"]
    g  = np.random.default_rng(seed)
    Zs = g.standard_normal((n, k)) @ model["L"].T
    Us = np.clip(stats.norm.cdf(Zs), 1e-9, 1 - 1e-9)
    sim = np.column_stack([stats.t.ppf(Us[:, j], *model["mar"][j]) for j in range(k)])
    return np.clip(sim, -model["cap"], model["cap"])

## Clustering helpers (unchanged from lecture)

In [ ]:
def codist(S):
    C = np.corrcoef(S, rowvar=False)
    return np.sqrt(0.5 * (1.0 - C))

def kmeans_(x, k, maxiters=100, tol=1e-5, seed=0):
    g  = np.random.default_rng(seed)
    N, d = len(x), len(x[0])
    distances  = np.zeros(N)
    assignment = g.integers(1, k + 1, size=N)
    reps = [np.zeros(d) for _ in range(k)]
    Jprev = np.inf
    for it in range(1, maxiters + 1):
        for j in range(1, k + 1):
            grp = np.where(assignment == j)[0]
            if len(grp) > 0:
                reps[j - 1] = np.mean([x[i] for i in grp], axis=0)
        for i in range(N):
            dd = [np.linalg.norm(x[i] - reps[j]) for j in range(k)]
            distances[i] = min(dd)
            assignment[i] = int(np.argmin(dd)) + 1
        J = np.linalg.norm(distances) ** 2 / N
        if it > 1 and abs(J - Jprev) < tol * J:
            break
        Jprev = J
    return assignment

# Cluster once, same for both gamma regimes
D      = codist(R)
labels = kmeans_([D[:, i] for i in range(p)], K, seed=RNG_SEED)
print("cluster sizes:", [int((labels == c).sum()) for c in range(1, K + 1)])

## Part (a) -- Build the three portfolios for both gamma regimes

Simulate once per portfolio and reuse across both gamma regimes so any cross-regime difference is purely an objective effect.

In [ ]:
# Simulate once, reused across both gamma
model_all = dict(mar=marginals_all, L=chol_all, cap=cap_all, k=p)
sim_all   = simulate(model_all, N_MC, seed=RNG_SEED)

# Within-cluster simulations for Portfolio A
within_sims  = {}
cluster_cols = {}
for c in range(1, K + 1):
    idx = np.where(labels == c)[0]
    if len(idx) == 0:
        continue
    mc = fit_gaussian_copula(R[:, idx])
    within_sims[c]  = simulate(mc, N_MC, seed=RNG_SEED + c)
    cluster_cols[c] = idx

print("Simulations ready.")

In [ ]:
results = {}

for gamma in [1, 3]:
    print(f"\n=== gamma = {gamma} ===")

    # Portfolio B: all-at-once
    w_B = crra_opt(sim_all, gamma)
    print(f"B: sum(w)={w_B.sum():.4f}  rf_share={1-w_B.sum():.4f}")

    # Portfolio A: NCO
    within_w      = {}
    cluster_train = []
    for c in sorted(cluster_cols):
        idx = cluster_cols[c]
        wc  = crra_opt(within_sims[c], gamma)
        within_w[c] = wc
        cluster_train.append(R[:, idx] @ wc)

    C_train  = np.column_stack(cluster_train)
    model_ac = fit_gaussian_copula(C_train)
    sim_ac   = simulate(model_ac, N_MC, seed=RNG_SEED + 99)
    w_across = crra_opt(sim_ac, gamma)

    w_A = np.zeros(p)
    for a_wt, c in zip(w_across, sorted(within_w)):
        idx = cluster_cols[c]
        w_A[idx] = a_wt * within_w[c]

    print(f"A: across-cluster weights = {np.round(w_across, 4)}")
    print(f"   within-cluster sums    = "
          f"{[round(float(within_w[c].sum()),4) for c in sorted(within_w)]}")
    print(f"   sum(w)={w_A.sum():.4f}  rf_share={1-w_A.sum():.4f}")

    # Portfolio 1/N: 1/20 each including risk-free
    w_1N = np.full(p, 1.0 / 20)

    results[gamma] = dict(w_A=w_A, w_B=w_B, w_1N=w_1N,
                          w_across=w_across, within_w=within_w)

In [ ]:
asset_names = [f"stock_{i+1}" for i in range(p)]

for gamma in [1, 3]:
    r = results[gamma]
    df_w = pd.DataFrame({
        "NCO (A)":         np.round(r["w_A"],  6),
        "All-at-once (B)": np.round(r["w_B"],  6),
        "1/N":             np.round(r["w_1N"], 6),
    }, index=asset_names)
    rf_row = pd.DataFrame({
        "NCO (A)":         [round(1 - r["w_A"].sum(),  6)],
        "All-at-once (B)": [round(1 - r["w_B"].sum(), 6)],
        "1/N":             [round(1 - r["w_1N"].sum(), 6)],
    }, index=["risk-free"])
    print(f"\n--- gamma = {gamma}: risky weights + rf share ---")
    print(pd.concat([df_w, rf_row]).to_string())

### Part (a) written summary

**gamma = 1.** Both NCO (A) and all-at-once (B) invest the entire budget in risky assets, leaving 0% in the risk-free. The weight concentrates almost entirely on stocks 3 and 14. NCO allocates 70.4% to stock 3 and 29.6% to stock 14. All-at-once allocates 78.1% to stock 3 and 21.9% to stock 14. The across-cluster weights for NCO are [0, 0, 1, 0, 0], putting 100% of the across-cluster budget into cluster 3 and zeroing out the other four. Within-cluster weight sums are [1.0, 0.505, 1.0, 1.0, 1.0], so all clusters except cluster 2 are fully invested. The 1/N benchmark holds 5% in each of the 19 risky assets and 5% in the risk-free.

**gamma = 3.** With greater risk aversion, both optimized portfolios spread more broadly. NCO holds stocks 3, 5, and 14 with weights 42.1%, 17.6%, and 27.2%, leaving 13.1% in the risk-free. All-at-once spreads across stocks 3, 5, 9, 11, 12, 14, and 19 with rf share near 0%. The across-cluster weights for NCO remain [0, 0, 1, 0, 0], so all risky budget still flows through cluster 3. The 1/N portfolio is unchanged across regimes.

## Part (b) -- Out-of-sample horse race

In [ ]:
W0 = 10_000

for gamma in [1, 3]:
    r = results[gamma]

    g_A  = gross_growth(r["w_A"],  R_test)
    g_B  = gross_growth(r["w_B"],  R_test)
    g_1N = gross_growth(r["w_1N"], R_test)

    V_A  = W0 * np.cumprod(np.maximum(1 + g_A,  EPS))
    V_B  = W0 * np.cumprod(np.maximum(1 + g_B,  EPS))
    V_1N = W0 * np.cumprod(np.maximum(1 + g_1N, EPS))

    lg_A  = np.log(np.maximum(1 + g_A,  EPS))
    lg_B  = np.log(np.maximum(1 + g_B,  EPS))
    lg_1N = np.log(np.maximum(1 + g_1N, EPS))

    days = range(1, T_test + 1)

    # (i) Wealth paths
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(days, V_A,  color="tab:blue",  label="NCO (A)")
    ax.plot(days, V_B,  color="tab:red",   label="All-at-once (B)")
    ax.plot(days, V_1N, color="tab:green", label="1/N")
    ax.set_title(f"Out-of-sample wealth paths, gamma = {gamma}")
    ax.set_xlabel("test day"); ax.set_ylabel("portfolio value ($)")
    ax.legend(); plt.tight_layout(); plt.show()

    # (ii) Log-growth histograms
    bins = np.linspace(min(lg_A.min(), lg_B.min(), lg_1N.min()),
                       max(lg_A.max(), lg_B.max(), lg_1N.max()), 50)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(lg_A,  bins=bins, alpha=0.5, color="tab:blue",
            label=f"NCO (mean {lg_A.mean():+.5f})")
    ax.hist(lg_B,  bins=bins, alpha=0.5, color="tab:red",
            label=f"All-at-once (mean {lg_B.mean():+.5f})")
    ax.hist(lg_1N, bins=bins, alpha=0.5, color="tab:green",
            label=f"1/N (mean {lg_1N.mean():+.5f})")
    for m, col in zip([lg_A.mean(), lg_B.mean(), lg_1N.mean()],
                      ["tab:blue", "tab:red", "tab:green"]):
        ax.axvline(m, color=col, ls="--", lw=1.2)
    ax.axvline(0, color="k", lw=0.6)
    ax.set_title(f"Realized daily log-growth, gamma = {gamma} (dashed = mean)")
    ax.set_xlabel(r"$\log(1+g_t)$"); ax.set_ylabel("# test days")
    ax.legend(); plt.tight_layout(); plt.show()

    # Summary table
    summary = pd.DataFrame({
        "rf share":             [1-r["w_A"].sum(),  1-r["w_B"].sum(),  1-r["w_1N"].sum()],
        "final wealth ($)":     [V_A[-1],  V_B[-1],  V_1N[-1]],
        "total return":         [V_A[-1]/W0-1, V_B[-1]/W0-1, V_1N[-1]/W0-1],
        "mean log-growth":      [lg_A.mean(),  lg_B.mean(),  lg_1N.mean()],
        "worst log-growth":     [lg_A.min(),   lg_B.min(),   lg_1N.min()],
        "realized CRRA E[u_g]": [realized_crra(r["w_A"],  R_test, gamma),
                                  realized_crra(r["w_B"],  R_test, gamma),
                                  realized_crra(r["w_1N"], R_test, gamma)],
    }, index=["NCO (A)", "All-at-once (B)", "1/N"])

    print(f"\n--- Summary table: gamma = {gamma} ---")
    print(summary.to_string(float_format=lambda x: f"{x:.5f}"))

## Part (c) -- Why the correlation distance?

MLdP maps $\rho_{ij}$ to $D_{ij} = \sqrt{\frac{1}{2}(1-\rho_{ij})}$ before clustering for three reasons.

**(i) Clustering needs a true metric; correlation is not one.** $k$-means operates in a space where closer means more similar and the triangle inequality holds. Correlation is a similarity in $[-1,1]$, not a distance: it can be negative, $\rho_{ii} = 1$ is the wrong identity, and $1-\rho$ fails the triangle inequality. $D_{ij}$ is a proper metric, with $D = 0 \iff \rho = 1$, $D = 1 \iff \rho = -1$, non-negative, symmetric, and satisfying the triangle inequality. Feeding $k$-means the distance columns gives a geometrically coherent input space.

**(ii) The sign problem.** On raw $\rho$, two assets with $\rho = -0.95$ (a near-perfect hedge, economically as unlike as possible) and two with $\rho = +0.95$ (near-perfect co-movers) are equidistant from zero and could end up in the same cluster. $D$ is monotone in the right direction: strong co-movers collapse toward $D \to 0$ and cluster together, while strong hedges push to $D \to 1$ and separate. The distance transform enforces the economically correct notion that hedges and co-movers belong on opposite ends of the similarity scale.

**(iii) Clustering on the full distance profile.** The feature vector for asset $i$ fed to $k$-means is its entire column $(D_{1i}, D_{2i}, \dots, D_{pi})$, its correlation-distance to every other asset. Two assets are judged similar when they have a similar pattern of co-movement with the whole universe, not merely when they happen to be correlated with each other. This is a richer, more noise-robust notion of co-grouping that is only coherent once $D$ is a genuine coordinate space, which raw $\rho$ is not.

## Part (d) -- Synthesis

**1. Does NCO still beat the all-at-once approach under no-short/no-borrow constraints?**

NCO does not meaningfully beat all-at-once once shorting and borrowing are disallowed. At $\gamma = 1$, NCO finishes at \$13,482 versus \$13,113 for all-at-once, a modest gap, while 1/N finishes at \$14,910, well ahead of both. The no-short constraint is particularly damaging to NCO's across-cluster step: the across-cluster weights collapse to [0, 0, 1, 0, 0], putting all weight on a single cluster and zeroing out the other four entirely. This means the nested structure provides no cross-cluster diversification at all. NCO's lecture-notebook edge came from leveraging across clusters and shorting within them. With those degrees of freedom removed, NCO becomes a single-cluster optimizer and the structural advantage disappears.

**2. How much do the optimizers leave in the risk-free asset, and how does that change with $\gamma$?**

At $\gamma = 1$, both NCO and all-at-once invest the full budget in risky assets, leaving effectively 0% in the risk-free. At $\gamma = 3$, NCO pulls back to a 13.1% risk-free share, while all-at-once still invests nearly the full budget. This is the expected direction: higher risk aversion shifts weight toward the safe asset. Notably, Portfolio B does not hold more cash at $\gamma = 3$ as might be expected. Instead the optimizer spreads across more risky assets, stocks 3, 5, 9, 11, 12, 14, and 19, rather than retreating to cash, suggesting the all-at-once solver finds enough diversification among risky assets to satisfy the higher penalty on downside.

**3. Does raising $\gamma$ from 1 to 3 reshuffle the ranking? How do the optimized portfolios compare to 1/N at $\gamma = 3$?**

The ranking does not change: 1/N finishes first at both $\gamma = 1$ and $\gamma = 3$, with NCO and all-at-once behind it in both regimes. At $\gamma = 3$, 1/N has a mean log-growth of +0.00080, compared to +0.00064 for NCO and +0.00068 for all-at-once. On the worst day, 1/N loses only 2.7%, versus 3.1% for NCO and 3.1% for all-at-once. The 1/N benchmark outperforms on both criteria. This is consistent with DeMiguel, Garlappi and Uppal (2009): once realistic constraints are imposed, estimation error in the optimizer's inputs erodes its advantage over naive equal-weighting. The 1/N portfolio makes no estimation bets and carries no model risk, which under the long-only constraint turns out to be a more robust strategy than fitting a copula and optimizing.